# 05 · Experimentos que eligieron la receta de v30 (opcional)

Usan la partición histórica (entrenamiento feb–jun + ago–oct de 2025, validación nov–dic para la parada temprana y
test ene + jul) y se comparan con las predicciones guardadas de V27 sobre los mismos vuelos.

1. Un CatBoost con las 137 variables base, y otro con las 16 de fuente añadidas.
2. La mezcla de clasificador de copia y regresor del rodaje (`scripts/experimentos/mezcla.py`).
3. La receta de v30: la mezcla con las 177 variables y lr 0,03. Su parada temprana fija los árboles del modelo
   final: clasificador ≈ 3.200 → 3.300, regresor ≈ 21.500 → 24.000.
4. La ablación (`scripts/experimentos/ablacion.py`, 34 entrenamientos, ~4 h); aquí solo se muestra su resumen.

Lleva ~1 h de GPU, y el cuaderno 00 solo lo ejecuta con `EXPERIMENTOS = 1`.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "cuaderno.py").exists())
sys.path.insert(0, str(ROOT / "scripts"))
from cuaderno import elegir_periodo, existen, interruptor, run  # noqa: E402

assert elegir_periodo("ranking") == "ranking", "05 usa la partición histórica de 2025: solo con PERIODO = 'ranking'"

In [ ]:
run("scripts/experimentos/experimento.py", "base")
run("scripts/experimentos/experimento.py", "fuentes")
run("scripts/experimentos/mezcla.py", "fuentes")

In [ ]:
# receta de v30 en la partición histórica: fija el número de árboles del modelo final
run("scripts/experimentos/mezcla.py", "fuentes,adsb,adsb_stand", '{"tag": "lr03", "learning_rate": 0.03, "iterations": 26000}')

In [ ]:
import json

import pandas as pd

filas = []
for p in sorted((ROOT / "Data" / "experimentos").glob("*/metricas.json")):
    m = json.loads(p.read_text(encoding="utf-8"))
    filas.append({"experimento": p.parent.name, "val": m["val"]["rmse"], "test": m["test"]["rmse"],
                  "árboles clasificador": m.get("it_clf"), "árboles regresor": m.get("it_reg", m.get("iteraciones"))})
pd.DataFrame(filas).set_index("experimento").round(2)

In [ ]:
# resumen guardado de la ablación (para repetirla: python scripts/experimentos/ablacion.py todo y después
# python scripts/experimentos/ablacion_resumen.py, que lo recalcula a partir de las predicciones de cada prueba)
pd.read_csv(ROOT / "Data" / "ablacion" / "resumen.csv", index_col=0).sort_values("d_val_normal").round(2)